# Module 06 — Lesson 01
# Handling Missing Values: Drop, Fill, or Impute?

---

> In Module 05 you learned the *mechanics* of missing data — `.isna()`, `.dropna()`, `.fillna()`. This lesson is about the *decision*: when a column has gaps, which of five strategies actually makes sense, and why picking the wrong one can quietly wreck a model three modules from now?

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Explain the difference between data missing *at random* and missing *for a reason* (MCAR / MAR / MNAR, in plain English)
- Choose between **deletion**, **statistical imputation**, **group-wise imputation**, and **flagging** for a given column
- Impute per-group values with `groupby().transform()`, and explain why an overall mean/median can be the wrong fill value
- Build a **missing indicator** column to preserve the signal that a value was missing in the first place
- Avoid the classic imputation mistake: computing fill values from the *whole* dataset before a train/test split

---
## Why This Isn't Just "Last Module Again"

Picture an HR analyst cleaning a spreadsheet of 60 employees before it goes into a salary-fairness report. Three columns have gaps: `age`, `salary`, and `performance_rating`. It's tempting to run `.fillna(mean)` on all three and move on. That would be a mistake, and not a small one:

- `age` is missing for a handful of people almost certainly **at random** — a form field someone forgot to fill in. Filling with the overall mean is safe.
- `salary` is missing more often for the **Sales** department specifically — reps are worse at submitting payroll self-reports. Filling with the *overall* mean would systematically overstate how much Sales actually earns, because you'd be blending in Engineering and Finance salaries.
- `performance_rating` is missing **because** the employee has been there less than a year and hasn't been reviewed yet. That's not random at all — it's structural. Filling it in with *any* number invents a rating that doesn't exist and hides a real, meaningful fact: "too new to rate."

This is the real skill of data cleaning: **the right fix for a missing value depends on *why* it's missing**, not just on the fact that it's missing. Statisticians group this into three categories — you don't need the formal names to use the idea, but they're useful shorthand:

| Type | Plain English | Example here | Safe strategy |
|---|---|---|---|
| **MCAR** — Missing Completely at Random | No pattern; pure chance | A few `age` values left blank | Drop or simple impute |
| **MAR** — Missing at Random (but related to another column) | The *chance* of missing depends on something else you can see | `salary` missing more in Sales | Group-wise impute |
| **MNAR** — Missing Not at Random | The missingness itself carries meaning | `performance_rating` missing = "not yet reviewed" | Flag it, don't fake it |

Let's load the messy dataset and work through all five strategies on it.

In [1]:
import pandas as pd
import numpy as np

employees = pd.read_csv("data/employees.csv")
print(f"employees.shape: {employees.shape}")
employees.head(12)

employees.shape: (60, 7)


,employee_id,name,department,age,years_experience,salary,performance_rating
0,1001,Rohan Pillai,HR,27.0,6.4,56900.0,4.0
1,1002,Riya Singh,Engineering,43.0,10.0,95300.0,5.0
2,1003,Karan Kapoor,Engineering,25.0,2.3,74400.0,3.0
3,1004,Rohan Bose,Marketing,46.0,13.6,85200.0,5.0
4,1005,Kabir Verma,HR,45.0,0.7,50700.0,NaN
5,1006,Pooja Chauhan,Not Provided,25.0,11.1,92200.0,5.0
6,1007,Suresh Pillai,Engineering,50.0,1.5,76200.0,5.0
7,1008,Tanvi Pillai,Engineering,37.0,2.7,79300.0,4.0
8,1009,Pooja Gupta,Sales,51.0,9.7,79600.0,4.0
9,1010,Simran Das,Engineering,23.0,0.5,70600.0,NaN


---
## 0. Quick Recap: Clean Up the Disguised Missing Value First

`department` has a placeholder string, `"Not Provided"`, that pandas won't auto-detect as missing (you saw this trick in Module 05, Lesson 06). One line converts it to a real `NaN` so every check below sees the full picture.

In [2]:
employees["department"] = employees["department"].replace("Not Provided", np.nan)

missing_counts = employees.isna().sum()
missing_pct = (employees.isna().mean() * 100).round(1)
pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})

,missing_count,missing_pct
employee_id,0,0.0
name,0,0.0
department,4,6.7
age,3,5.0
years_experience,0,0.0
salary,7,11.7
performance_rating,11,18.3


---
## 1. Strategy One: Deletion

Deletion is the right call when a column is missing a *small* percentage of values *at random*, or when a row is missing so many fields it's not usable at all. `department` is only missing 6.7% of the time with no obvious pattern — safe to drop those rows outright.

In [3]:
before = employees.shape[0]
employees_deletion_demo = employees.dropna(subset=["department"])
after = employees_deletion_demo.shape[0]
print(f"Dropping rows with missing department: {before} -> {after} rows ({before - after} dropped)")

Dropping rows with missing department: 60 -> 56 rows (4 dropped)


---
## 2. Strategy Two: Simple Statistical Imputation

For `age`, missingness looks like pure chance (MCAR) — nothing distinguishes the three people who left it blank. The median is a safe, simple fill value, since it isn't dragged around by a couple of unusually young or old employees the way a mean would be.

> **Looking ahead:** starting in Module 10 (ML Fundamentals) you'll switch to a library called `scikit-learn`, which wraps this exact idea — "learn a fill value, then apply it" — in a reusable object called `SimpleImputer`. The reason that matters: you `fit` it once on training data and `transform` other data with that *same* saved value, instead of recomputing the median every time. For now, plain pandas does the same job while you're still working with one dataset at a time.

In [4]:
age_median = employees["age"].median()
employees["age"] = employees["age"].fillna(age_median)

print(f"Median age used to fill gaps: {age_median}")
print(f"Missing values remaining in age: {employees['age'].isna().sum()}")

Median age used to fill gaps: 39.0
Missing values remaining in age: 0


---
## 3. Strategy Three: Group-Wise Imputation

`salary` is different: it's missing more often *within* the Sales department (MAR — the pattern depends on another column we can see). A single overall median would drag every Sales salary estimate toward the Engineering/Finance average. Instead, fill each missing salary with the **median of its own department** using `groupby().transform()`.

In [5]:
# transform() returns a Series the same shape/index as the original column,
# with each group's missing values replaced by that group's own median
dept_median_salary = employees.groupby("department")["salary"].transform("median")

employees["salary"] = employees["salary"].fillna(dept_median_salary)

print(f"Missing values remaining in salary: {employees['salary'].isna().sum()}")
print()
print("Median salary by department (what each gap got filled with):")
print(employees.groupby("department")["salary"].median().round(0))

Missing values remaining in salary: 0

Median salary by department (what each gap got filled with):
department
Engineering    80100.0
Finance        70050.0
HR             51400.0
Marketing      62350.0
Sales          57000.0
Name: salary, dtype: float64


---
## 4. Strategy Four: Flagging Instead of Faking

`performance_rating` is MNAR — it's missing *because* the employee is too new to have been reviewed. There is no honest number to put there. Imputing it (even with something "neutral" like the average rating) would erase a true fact about the employee and could bias any later analysis of performance.

The right move: leave the value missing (or drop it from rating-specific analysis), but add a **missing indicator column** so the fact of "not yet rated" survives as usable information.

In [6]:
employees["rating_missing"] = employees["performance_rating"].isna().astype(int)

print("New employees (rating_missing == 1) vs. rated employees, by years of experience:")
print(employees.groupby("rating_missing")["years_experience"].describe()[["count", "mean", "max"]])

New employees (rating_missing == 1) vs. rated employees, by years of experience:
                count      mean   max
rating_missing                       
0                49.0  4.995918  13.9
1                11.0  0.545455   0.8


Notice the pattern the indicator reveals: every `rating_missing == 1` employee has under a year of experience. That's exactly the kind of structural relationship you'd lose forever if you'd silently filled the column with a guessed number instead.

---
## 5. Putting the Decision Framework Together

| Column | Missingness pattern | Strategy chosen | Why |
|---|---|---|---|
| `department` | MCAR, small % | Drop rows | Too little signal to bother imputing a category |
| `age` | MCAR | Median impute (`SimpleImputer`) | Random gap, safe to fill with a robust central value |
| `salary` | MAR (depends on department) | Group-wise median impute | Overall median would bias Sales salaries downward/upward |
| `performance_rating` | MNAR (depends on tenure) | Flag, don't fill | The missingness itself is the useful information |

This is the mental checklist to run on *any* column with gaps: **how much is missing, and does the missingness correlate with anything?** The answer picks the strategy — not habit, not whichever pandas method you remember first.

---
## ⚠️ Common Mistakes

In [7]:
# -- Mistake 1: Imputing with the OVERALL mean/median when a clear subgroup pattern exists ----

raw = pd.read_csv("data/employees.csv")
raw["department"] = raw["department"].replace("Not Provided", np.nan)

overall_median = raw["salary"].median()
sales_median = raw[raw["department"] == "Sales"]["salary"].median()

print("Mistake 1 — filling every missing salary with the overall median ignores that")
print("Sales salaries run structurally lower/different from other departments:")
print(f"  Overall median salary: {overall_median:,.0f}")
print(f"  Sales-only median salary: {sales_median:,.0f}")
print("  -> Use groupby().transform('median') instead of a single global fillna() value.")

Mistake 1 — filling every missing salary with the overall median ignores that
Sales salaries run structurally lower/different from other departments:
  Overall median salary: 70,700
  Sales-only median salary: 57,000
  -> Use groupby().transform('median') instead of a single global fillna() value.


In [8]:
# -- Mistake 2: Computing the fill value from ALL the data, including rows that should ------
# --             have been held out as a test set --------------------------------------------

demo = raw.dropna(subset=["salary"]).copy()  # pretend this is fully observed for the demo

# A simple 80/20 split done by hand with pandas, just to illustrate the ordering problem
# (you'll use a dedicated train/test split tool once you reach Module 10)
train_demo = demo.sample(frac=0.8, random_state=1)
test_demo = demo.drop(train_demo.index)

wrong_median = demo["salary"].median()          # computed from train + test combined
right_median = train_demo["salary"].median()    # computed from train only

print("Mistake 2 — computing a fill value from the WHOLE dataset, then splitting, lets")
print("information from the test rows leak into training (the model indirectly 'sees the future'):")
print(f"  Median computed from ALL rows (wrong):   {wrong_median:,.0f}")
print(f"  Median computed from train rows only:    {right_median:,.0f}")
print()
print("The values may look close here, but the principle holds regardless of the gap:")
print("always compute fill statistics AFTER splitting, from the training portion only.")
print("You'll build this properly with real train/test splits starting in Module 10 —")
print("the habit of 'learn from train, apply to everything' starts here.")

Mistake 2 — computing a fill value from the WHOLE dataset, then splitting, lets
information from the test rows leak into training (the model indirectly 'sees the future'):
  Median computed from ALL rows (wrong):   70,700
  Median computed from train rows only:    69,000

The values may look close here, but the principle holds regardless of the gap:
always compute fill statistics AFTER splitting, from the training portion only.
You'll build this properly with real train/test splits starting in Module 10 —
the habit of 'learn from train, apply to everything' starts here.


In [9]:
# -- Mistake 3: Treating a structurally missing value (MNAR) as if it were random ------------

print("Mistake 3 — filling performance_rating with the average rating would invent a")
print("score for employees who were never reviewed, and silently erase the 'too new to")
print("rate' signal that the missing indicator preserved:")
fake_filled = raw["performance_rating"].fillna(raw["performance_rating"].mean())
print(f"  Employees who'd now show a rating that was never actually given: "
      f"{raw['performance_rating'].isna().sum()}")
print("  -> Keep the NaN (or drop from rating analysis) and rely on the flag column instead.")

Mistake 3 — filling performance_rating with the average rating would invent a
score for employees who were never reviewed, and silently erase the 'too new to
rate' signal that the missing indicator preserved:
  Employees who'd now show a rating that was never actually given: 11
  -> Keep the NaN (or drop from rating analysis) and rely on the flag column instead.


---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Diagnose Before You Fix

Load `data/employees.csv` fresh, clean up the `"Not Provided"` placeholder in `department`, and print the missing count and missing percentage for every column.

In [10]:
# Your code here

### Exercise 2 — Classify the Missingness

For each of `age`, `salary`, and `performance_rating`, write a one-line comment stating whether you'd call it MCAR, MAR, or MNAR, and why — you already have the reasoning from this lesson, restate it in your own words.

In [11]:
# Your code here (as comments)

### Exercise 3 — Group-Wise Impute a Different Column

Impute missing `age` values using the **median age within each department** instead of the overall median from Section 2. Confirm no missing values remain.

In [12]:
# Your code here

### Exercise 4 — Build Your Own Missing Indicator

Create a new column `salary_was_missing` that is `1` where `salary` was originally missing and `0` otherwise, using the *raw* (unfilled) data. Then check whether `salary_was_missing == 1` rows are concentrated in a particular department.

In [13]:
# Your code here

### Exercise 5 — (Challenge) Compare Two Imputation Strategies

Fill missing `salary` values two ways: (a) the overall median, (b) the department-wise median. Compare the resulting mean salary *per department* for each approach and explain in a comment which one distorts the Sales department's numbers more, and why.

In [14]:
# Your code here

---
## 📌 Key Takeaways

- **The right fix depends on *why* a value is missing, not just that it is** — random gaps (MCAR) tolerate simple imputation, gaps tied to another column (MAR) need group-wise imputation, and gaps that are themselves meaningful (MNAR) should be flagged, never faked.
- **`SimpleImputer` and `groupby().transform()` beat ad-hoc `.fillna()` calls once cleaning is part of a repeatable pipeline** — an imputer object remembers what it learned and can be reapplied consistently.
- **Always fit imputation statistics on training data only** — computing a fill value from data that includes your future test set is a data leakage bug that silently inflates model performance later.